In [ ]:
import os
import cv2
import numpy as np

# ============================================================
ROOT_DIR = r"D:\dataset\PCBData\PCBData\group00041"      # 数据集根目录，将递归搜索所有test图片
# ROOT_DIR = r"D:\dataset\PCBData\1"
OUTPUT_DIR = r"./output"                       # 结果图片与统计报告输出目录
IOU_THRESHOLD = 0.5                           # IoU匹配阈值（大于等于此值且类别相同视为正确检出）

# 可视化颜色（BGR格式）
COLOR_RED = (0, 0, 255)       # 漏检（未匹配的标注框）
COLOR_BLUE = (255, 0, 0)      # 正确检出（匹配的检测框）
COLOR_YELLOW = (0, 255, 255)  # 误检（未匹配的检测框）
# ============================================================


In [ ]:

# ------------------------- 工具函数 -------------------------
def compute_iou(box1, box2):
    """
    计算两个矩形框的IoU
    box: [x1, y1, x2, y2]  (左上角与右下角坐标)
    """
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter_area = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = area1 + area2 - inter_area
    if union_area <= 0:
        return 0.0
    return inter_area / union_area


def read_gt(txt_path):
    """
    读取标注txt文件，每行格式：x1 y1 x2 y2 class
    返回标注列表，每个元素为 {'bbox': [x1,y1,x2,y2], 'class': int}
    若文件不存在则返回空列表
    """
    gts = []
    if not os.path.exists(txt_path):
        return gts
    with open(txt_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                x1, y1, x2, y2, cls = map(int, parts)
                gts.append({'bbox': [x1, y1, x2, y2], 'class': cls})
    return gts

import numpy as np

# ============================================================
MATCH_MODE = 'center_overlap'   # 可选: 'iou', 'center_overlap', 'center_only'
IOU_THRESHOLD = 0.5             # 仅当 MATCH_MODE='iou' 时使用
CENTER_DIST_THRESH = 20.0       # 中心距离阈值（像素），仅当 center_overlap 或 center_only 时使用
OVERLAP_RATIO_DET = 0.3         # 重叠面积占检测框面积的最小比例
OVERLAP_RATIO_GT = 0.3          # 重叠面积占标注框面积的最小比例
# ============================================================

def match_detections(dets, gts, mode='center_overlap', iou_thresh=0.5,
                     center_dist_thresh=20.0, overlap_ratio_det=0.3, overlap_ratio_gt=0.3):
    """
    匹配检测框与标注框，支持多种匹配准则。
    匹配条件：类别相同 或 属于同一类型组（1,3,6 一组；2,4,5 一组）。
    返回: matches (det_idx, gt_idx), unmatched_dets, unmatched_gts
    """
    if not dets or not gts:
        return [], list(range(len(dets))), list(range(len(gts)))

    # 类型分组函数
    def get_group(cls):
        if cls in (1, 3, 6):
            return 0
        elif cls in (2, 4, 5):
            return 1
        else:
            return None  # 未分组类别视为独立（不与任何组兼容）

    def compatible(cls1, cls2):
        if cls1 == cls2:
            return True
        g1 = get_group(cls1)
        g2 = get_group(cls2)
        return g1 is not None and g1 == g2

    def compute_score(det, gt):
        # 只有类型兼容才计算重叠得分
        if not compatible(det['class'], gt['class']):
            return 0.0

        b1 = det['bbox']
        b2 = gt['bbox']
        x1 = max(b1[0], b2[0])
        y1 = max(b1[1], b2[1])
        x2 = min(b1[2], b2[2])
        y2 = min(b1[3], b2[3])
        inter = max(0, x2 - x1) * max(0, y2 - y1)
        if inter == 0:
            return 0.0

        if mode == 'iou':
            area1 = (b1[2]-b1[0]) * (b1[3]-b1[1])
            area2 = (b2[2]-b2[0]) * (b2[3]-b2[1])
            union = area1 + area2 - inter
            return inter / union if union > 0 else 0.0

        elif mode == 'center_overlap':
            cx1, cy1 = (b1[0]+b1[2])/2, (b1[1]+b1[3])/2
            cx2, cy2 = (b2[0]+b2[2])/2, (b2[1]+b2[3])/2
            dist = np.hypot(cx1-cx2, cy1-cy2)
            if dist > center_dist_thresh:
                return 0.0
            area1 = (b1[2]-b1[0]) * (b1[3]-b1[1])
            area2 = (b2[2]-b2[0]) * (b2[3]-b2[1])
            if inter / area1 < overlap_ratio_det and inter / area2 < overlap_ratio_gt:
                return 0.0
            return inter

        elif mode == 'center_only':
            cx1, cy1 = (b1[0]+b1[2])/2, (b1[1]+b1[3])/2
            cx2, cy2 = (b2[0]+b2[2])/2, (b2[1]+b2[3])/2
            dist = np.hypot(cx1-cx2, cy1-cy2)
            if dist <= center_dist_thresh:
                return 1.0
            return 0.0

        else:
            raise ValueError(f"未知匹配模式: {mode}")

    # 构建得分矩阵
    score_mat = np.zeros((len(dets), len(gts)))
    for i, det in enumerate(dets):
        for j, gt in enumerate(gts):
            score_mat[i, j] = compute_score(det, gt)

    # 贪心匹配（按得分降序）
    pairs = [(i, j) for i in range(len(dets)) for j in range(len(gts)) if score_mat[i, j] > 0]
    pairs.sort(key=lambda x: score_mat[x[0], x[1]], reverse=True)

    used_det = set()
    used_gt = set()
    matches = []
    for i, j in pairs:
        if i not in used_det and j not in used_gt:
            matches.append((i, j))
            used_det.add(i)
            used_gt.add(j)

    unmatched_dets = [i for i in range(len(dets)) if i not in used_det]
    unmatched_gts = [j for j in range(len(gts)) if j not in used_gt]
    return matches, unmatched_dets, unmatched_gts

# ------------------------- 路径构造函数 -------------------------
def find_test_images(root):
    """递归查找所有以 '_test.jpg' 结尾的图片路径"""
    test_images = []
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if f.endswith('_test.jpg'):
                test_images.append(os.path.join(dirpath, f))
    return test_images


def construct_temp_path(test_path):
    """根据test图片路径构造对应的temp图片路径"""
    return test_path.replace('_test.jpg', '_temp.jpg')


def construct_txt_path(test_path):
    """
    根据test图片路径构造对应的txt标注文件路径
    规则：图片位于 .../groupXXXXX/YYYYY/ 下，txt位于 .../groupXXXXX/YYYYY_not/ 下，文件名相同（不含后缀）
    """
    dirname = os.path.dirname(test_path)           # .../groupXXXXX/YYYYY
    basename = os.path.basename(test_path)         # YYYYYYY_test.jpg
    base_no_ext = basename.replace('_test.jpg', '')  # YYYYYYY
    parent_dir = os.path.dirname(dirname)          # .../groupXXXXX
    folder_name = os.path.basename(dirname)        # YYYYY
    txt_dir = os.path.join(parent_dir, folder_name + '_not')
    txt_name = base_no_ext + '.txt'
    return os.path.join(txt_dir, txt_name)



# 分类

In [ ]:
# ==========================================
# 缺陷通用分类逻辑（此函数不准确，后续需要优化）
# ==========================================
def classify_defects(cu_candidates, bg_candidates, context):
    """
    输入: 候选轮廓列表, 连通域上下文
    输出: [{'bbox': [x1,y1,x2,y2], 'class': int}, ...]
    类别映射: 1-开路, 2-短路, 3-缺口, 4-凸铜, 5-残铜, 6-针孔
    """
    results = []
    h, w = context['shape']
    
    ref_cu_labels = context['ref_cu_labels']
    ref_bg_labels = context['ref_bg_labels']
    test_cu_labels = context['test_cu_labels']
    test_bg_labels = context['test_bg_labels']

    # Tolerance + 1.0 (2.5) 与 2.0 近似使用 5x5 核
    kernel_5x5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

    def get_bbox(cnt):
        x, y, w_box, h_box = cv2.boundingRect(cnt)
        return [x, y, x + w_box, y + h_box]

    def check_geometry(cnt):
        # 计算圆度 (circularity)
        area = cv2.contourArea(cnt)
        perimeter = cv2.arcLength(cnt, True)
        circ = (4 * np.pi * area) / (perimeter * perimeter) if perimeter > 0 else 0
        
        # 计算非等轴度 (anisometry)
        rect = cv2.minAreaRect(cnt)
        w_rect, h_rect = rect[1]
        anisometry = max(w_rect, h_rect) / min(w_rect, h_rect) if min(w_rect, h_rect) > 0 else 1.0
            
        return (0.1 <= circ <= 1.0) and (1.0 <= anisometry <= 3.0)
    def check_geometry_and_fatness(mask, cnt):
        """
        引入基于距离变换的最大内接圆半径检测，覆盖复合边缘噪声场景。
        """
        # 1. 提取最大内接圆半径 (数学上等价于 HALCON 的 inner_circle)
        dist_transform = cv2.distanceTransform(mask, cv2.DIST_L2, 5)
        _, radius, _, _ = cv2.minMaxLoc(dist_transform)
        
        # 2. 提取长宽比和圆度
        area = cv2.contourArea(cnt)
        perimeter = cv2.arcLength(cnt, True)
        circ = (4 * np.pi * area) / (perimeter * perimeter) if perimeter > 0 else 0
        
        rect = cv2.minAreaRect(cnt)
        w_rect, h_rect = rect[1]
        anisometry = max(w_rect, h_rect) / min(w_rect, h_rect) if min(w_rect, h_rect) > 0 else 1.0
            
        # “胖度特权”覆盖逻辑
        if radius >= 2.5:
            return True
        elif anisometry <= 3.0 and circ >= 0.1:
            return True
            
        return False
    # 1. 铜层候选分类 (短路、凸铜、残铜)
    for cnt in cu_candidates:
        mask = np.zeros((h, w), dtype=np.uint8)
        cv2.drawContours(mask, [cnt], -1, 255, -1)
        dilated = cv2.dilate(mask, kernel_5x5)
        
        # 统计与 RefCu 的有效接触数量
        overlaps = np.unique(ref_cu_labels[dilated > 0])
        num_valid = 0
        for label in overlaps:
            if label == 0: continue
            inter_area = np.count_nonzero((ref_cu_labels == label) & (dilated > 0))
            if inter_area >= 3:
                num_valid += 1
        
        bbox = get_bbox(cnt)

        if num_valid >= 2:
            results.append({'bbox': bbox, 'class': 2}) # 线间短路
        elif num_valid == 1:
            # 单接触：利用背景连通域变化区分单线短路和凸铜
            dilated_for_bg = cv2.dilate(mask, kernel_5x5)
            num_bg_test = len(set(test_bg_labels[dilated_for_bg > 0]) - {0})
            num_bg_ref = len(set(ref_bg_labels[dilated_for_bg > 0]) - {0})
            
            if num_bg_test - num_bg_ref >= 1:
                results.append({'bbox': bbox, 'class': 2}) # 单线短路
            else:
                # if check_geometry(cnt):
                if check_geometry_and_fatness(mask, cnt):
                    results.append({'bbox': bbox, 'class': 4}) # 凸铜
        elif num_valid == 0:
            results.append({'bbox': bbox, 'class': 5}) # 残铜 (完全无接触)

    # 2. 背景层候选分类 (开路、缺口、针孔)
    for cnt in bg_candidates:
        mask = np.zeros((h, w), dtype=np.uint8)
        cv2.drawContours(mask, [cnt], -1, 255, -1)
        dilated = cv2.dilate(mask, kernel_5x5)
        
        # 统计与 RefBg 的有效接触数量
        overlaps = np.unique(ref_bg_labels[dilated > 0])
        num_valid = 0
        for label in overlaps:
            if label == 0: continue
            inter_area = np.count_nonzero((ref_bg_labels == label) & (dilated > 0))
            if inter_area >= 3:
                num_valid += 1
        
        bbox = get_bbox(cnt)

        if num_valid >= 2:
            results.append({'bbox': bbox, 'class': 1}) # 线间开路
        elif num_valid == 1:
            # 单接触：利用铜连通域变化区分单线开路和缺失
            dilated_for_cu = cv2.dilate(mask, kernel_5x5)
            num_cu_test = len(set(test_cu_labels[dilated_for_cu > 0]) - {0})
            num_cu_ref = len(set(ref_cu_labels[dilated_for_cu > 0]) - {0})
            
            if num_cu_test - num_cu_ref >= 1:
                results.append({'bbox': bbox, 'class': 1}) # 单线开路
            else:
                # if check_geometry(cnt):
                if check_geometry_and_fatness(mask, cnt):
                    results.append({'bbox': bbox, 'class': 3}) # 缺口/缺失
        elif num_valid == 0:
            results.append({'bbox': bbox, 'class': 6}) # 针孔 (完全无接触)

    return results

In [ ]:
def align_images_coarse(img_test, img_ref):
    """
    输入: 测试图 (numpy array), 参考图 (numpy array)
    输出: 经过仿射变换配准并裁剪后的测试图 (numpy array)
    """
    DOWNSAMPLE_RATIO = 0.25
    RANSAC_REPROJ_THRESHOLD = 3.0

    small_ref = cv2.resize(img_ref, (0, 0), fx=DOWNSAMPLE_RATIO, fy=DOWNSAMPLE_RATIO, interpolation=cv2.INTER_AREA)
    small_test = cv2.resize(img_test, (0, 0), fx=DOWNSAMPLE_RATIO, fy=DOWNSAMPLE_RATIO, interpolation=cv2.INTER_AREA)

    sift = cv2.SIFT_create()
    kp_ref, des_ref = sift.detectAndCompute(small_ref, None)
    kp_test, des_test = sift.detectAndCompute(small_test, None)

    if des_ref is None or des_test is None:
        raise ValueError("特征提取失败。")

    bf = cv2.BFMatcher(cv2.NORM_L2)
    matches = bf.knnMatch(des_test, des_ref, k=2)

    good_matches = []
    for m, n in matches:
        if m.distance < 0.75 * n.distance:
            good_matches.append(m)

    if len(good_matches) < 3:
        raise ValueError("匹配点过少，无法计算初始矩阵。")

    src_pts = np.float32([kp_test[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp_ref[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

    # 计算包含拉伸的 6 自由度仿射变换矩阵
    matrix_small, _ = cv2.estimateAffine2D(src_pts, dst_pts, method=cv2.RANSAC, ransacReprojThreshold=RANSAC_REPROJ_THRESHOLD)

    if matrix_small is None:
        raise ValueError("无法计算粗略仿射变换矩阵。")

    warp_matrix = matrix_small.copy()
    warp_matrix[0, 2] /= DOWNSAMPLE_RATIO
    warp_matrix[1, 2] /= DOWNSAMPLE_RATIO

    h, w = img_ref.shape
    # 对测试图进行变换，并自动匹配参考图的尺寸 (w, h)
    img_test_aligned = cv2.warpAffine(img_test, warp_matrix, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    # cv2.imwrite("aligned_result.jpg", img_test_aligned)   # 保存为 JPG 格式，文件名可自定义
    # raise ValueError("无法计算粗略仿射变换矩阵。")
    return img_test_aligned

# 各类检测算法定义

## 区域

In [ ]:
def extract_defect_candidates(img_test, img_ref):
    """
    输入: 测试图, 模板图
    输出: 铜层异常轮廓, 背景层异常轮廓, 供分类使用的连通域上下文
    """

    # 1. 提取铜区域 (0-127) 和背景区域 (128-255)
    _, test_cu = cv2.threshold(img_test, 127, 255, cv2.THRESH_BINARY_INV)
    _, ref_cu = cv2.threshold(img_ref, 127, 255, cv2.THRESH_BINARY_INV)
    
    _, test_bg = cv2.threshold(img_test, 127, 255, cv2.THRESH_BINARY)
    _, ref_bg = cv2.threshold(img_ref, 127, 255, cv2.THRESH_BINARY)

    # 2. 构建容差带 (Tolerance = 1.5，对应 3x3 椭圆核)
    kernel_3x3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    ref_cu_max = cv2.dilate(ref_cu, kernel_3x3)
    ref_bg_max = cv2.dilate(ref_bg, kernel_3x3)

    # 3. 提取额外区域 (包含残铜和针孔)
    extra_cu_raw = cv2.bitwise_and(test_cu, cv2.bitwise_not(ref_cu_max))
    extra_bg_raw = cv2.bitwise_and(test_bg, cv2.bitwise_not(ref_bg_max))

    # 4. 过滤面积并打散连通域
    def filter_contours(mask):
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        return [cnt for cnt in contours if cv2.contourArea(cnt) >= 5]

    cu_candidates = filter_contours(extra_cu_raw)
    bg_candidates = filter_contours(extra_bg_raw)

    # 生成分类阶段所需的标签上下文
    _, ref_cu_labels = cv2.connectedComponents(ref_cu)
    _, ref_bg_labels = cv2.connectedComponents(ref_bg)
    _, test_cu_labels = cv2.connectedComponents(test_cu)
    _, test_bg_labels = cv2.connectedComponents(test_bg)

    context = {
        'shape': img_test.shape,
        'ref_cu_labels': ref_cu_labels,
        'ref_bg_labels': ref_bg_labels,
        'test_cu_labels': test_cu_labels,
        'test_bg_labels': test_bg_labels
    }

    return cu_candidates, bg_candidates, context

## 骨架

In [ ]:
def extract_defect_candidates_skeleton(img_test, img_ref):

    # 1. 图像读取与二值化 (0-127为暗特征铜面)
    _, region_ref = cv2.threshold(img_ref, 127, 255, cv2.THRESH_BINARY_INV)
    _, region_test = cv2.threshold(img_test, 127, 255, cv2.THRESH_BINARY_INV)

    # 2. 提取铜面骨架 (依赖 opencv-contrib-python 中的 ximgproc)
    skeleton_test = cv2.ximgproc.thinning(region_test)

    # 3. 铜面骨架差异比对 (容差1像素，对应 3x3 核)
    kernel_1px = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    region_ref_tolerance = cv2.dilate(region_ref, kernel_1px)
    
    # difference (SkeletonTest, RegionRefTolerance, ExtraCopperRaw)
    extra_copper_raw = cv2.bitwise_and(skeleton_test, cv2.bitwise_not(region_ref_tolerance))

    # 4. 背景检测
    background_ref = cv2.bitwise_not(region_ref)
    background_test = cv2.bitwise_not(region_test)

    # 提取背景骨架
    skeleton_bg_test = cv2.ximgproc.thinning(background_test)

    # 背景差异比对 (容差1像素)
    bg_ref_dil = cv2.dilate(background_ref, kernel_1px)
    extra_bg_raw = cv2.bitwise_and(skeleton_bg_test, cv2.bitwise_not(bg_ref_dil))

    # 5. 为了兼容原分类器中的几何过滤（圆度要求面积非0），将1像素的骨架残段膨胀回实体块
    kernel_restore = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    extra_copper_blobs = cv2.dilate(extra_copper_raw, kernel_restore)
    extra_bg_blobs = cv2.dilate(extra_bg_raw, kernel_restore)

    # 过滤微小噪点 (连通域面积 >= 2)
    def get_filtered_contours(mask):
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        # 此处已经过膨胀，原先2像素长的骨架大约膨胀为面积 >= 10 的块
        return [cnt for cnt in contours if cv2.contourArea(cnt) >= 10]

    cu_candidates = get_filtered_contours(extra_copper_blobs)
    bg_candidates = get_filtered_contours(extra_bg_blobs)

    # 生成分类阶段所需的标签上下文
    _, ref_cu_labels = cv2.connectedComponents(region_ref)
    _, ref_bg_labels = cv2.connectedComponents(background_ref)
    _, test_cu_labels = cv2.connectedComponents(region_test)
    _, test_bg_labels = cv2.connectedComponents(background_test)

    context = {
        'shape': img_test.shape,
        'ref_cu_labels': ref_cu_labels,
        'ref_bg_labels': ref_bg_labels,
        'test_cu_labels': test_cu_labels,
        'test_bg_labels': test_bg_labels
    }

    return cu_candidates, bg_candidates, context

## 形态学

In [ ]:

def extract_defect_candidates_drc(img_test, img_ref):
    """
    形态学 DRC 检测 (灰度统计优化版)
    输出与原分类函数完全兼容的 cu_candidates, bg_candidates 和 context
    """

    h, w = img_test.shape

    # 1. 极性确认：二值化
    _, test_cu = cv2.threshold(img_test, 127, 255, cv2.THRESH_BINARY_INV)
    _, test_bg = cv2.threshold(img_test, 127, 255, cv2.THRESH_BINARY)
    _, ref_cu = cv2.threshold(img_ref, 127, 255, cv2.THRESH_BINARY_INV)
    _, ref_bg = cv2.threshold(img_ref, 127, 255, cv2.THRESH_BINARY)

    # 设定形态学内核 (Halcon 半径 5 对应 11x11, 半径 2 对应 5x5)
    kernel_r5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    kernel_r2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

    # ==========================================
    # 2. 提取初始异常，并用参考图屏蔽合法锐角
    # ==========================================
    # 铜层异常
    test_cu_opened = cv2.morphologyEx(test_cu, cv2.MORPH_OPEN, kernel_r5)
    cu_raw = cv2.bitwise_and(test_cu, cv2.bitwise_not(test_cu_opened))
    
    ref_cu_opened = cv2.morphologyEx(ref_cu, cv2.MORPH_OPEN, kernel_r5)
    ref_cu_raw = cv2.bitwise_and(ref_cu, cv2.bitwise_not(ref_cu_opened))
    ref_cu_dilated = cv2.dilate(ref_cu_raw, kernel_r2)
    cu_violations = cv2.bitwise_and(cu_raw, cv2.bitwise_not(ref_cu_dilated))

    # 背景层异常
    test_bg_opened = cv2.morphologyEx(test_bg, cv2.MORPH_OPEN, kernel_r5)
    bg_raw = cv2.bitwise_and(test_bg, cv2.bitwise_not(test_bg_opened))
    
    ref_bg_opened = cv2.morphologyEx(ref_bg, cv2.MORPH_OPEN, kernel_r5)
    ref_bg_raw = cv2.bitwise_and(ref_bg, cv2.bitwise_not(ref_bg_opened))
    ref_bg_dilated = cv2.dilate(ref_bg_raw, kernel_r2)
    bg_violations = cv2.bitwise_and(bg_raw, cv2.bitwise_not(ref_bg_dilated))

    # ==========================================
    # 3. 切除图像边缘区域的误检 (EdgeMargin = 10)
    # ==========================================
    margin = 10
    roi = np.zeros((h, w), dtype=np.uint8)
    cv2.rectangle(roi, (margin, margin), (w - margin - 1, h - margin - 1), 255, -1)
    
    cu_valid = cv2.bitwise_and(cu_violations, roi)
    bg_valid = cv2.bitwise_and(bg_violations, roi)

    # ==========================================
    # 4. 全局合并与“大吞小”灰度统计逻辑
    # ==========================================
    # 合并并执行闭运算桥接
    all_valid_raw = cv2.bitwise_or(cu_valid, bg_valid)
    all_closed = cv2.morphologyEx(all_valid_raw, cv2.MORPH_CLOSE, kernel_r5)

    # 提取连通域
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(all_closed, connectivity=8)

    # 构建灰度权重掩码 (int16 允许负数)
    weight_mask = np.zeros((h, w), dtype=np.int16)
    weight_mask[cu_valid > 0] = 1
    weight_mask[bg_valid > 0] = -1

    # 初始化用于存放最终分类簇的二值图
    cu_dominant_mask = np.zeros((h, w), dtype=np.uint8)
    bg_dominant_mask = np.zeros((h, w), dtype=np.uint8)

    # 遍历连通域进行向量化极速分类
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] >= 8:
            comp_mask = (labels == i)
            # 计算该连通域在权重图上的均值
            mean_val = np.mean(weight_mask[comp_mask])
            
            # 大吞小判定
            if mean_val >= 0:
                cu_dominant_mask[comp_mask] = 255
            else:
                bg_dominant_mask[comp_mask] = 255

    # ==========================================
    # 5. 提取轮廓并生成分类上下文
    # ==========================================
    # 注意：这里直接对闭运算桥接后的 Mask 提取轮廓，保证簇不被物理打散
    cu_candidates, _ = cv2.findContours(cu_dominant_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    bg_candidates, _ = cv2.findContours(bg_dominant_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # 准备分类器所需的上下文
    _, ref_cu_labels = cv2.connectedComponents(ref_cu)
    _, ref_bg_labels = cv2.connectedComponents(ref_bg)
    _, test_cu_labels = cv2.connectedComponents(test_cu)
    _, test_bg_labels = cv2.connectedComponents(test_bg)

    context = {
        'shape': (h, w),
        'ref_cu_labels': ref_cu_labels,
        'ref_bg_labels': ref_bg_labels,
        'test_cu_labels': test_cu_labels,
        'test_bg_labels': test_bg_labels
    }

    return cu_candidates, bg_candidates, context

## 轮廓

In [ ]:
def extract_defect_candidates_seed(img_test, img_ref):
    # 1. 提取铜区域 (0-127) 和 背景区域 (128-255)
    _, ref_cu = cv2.threshold(img_ref, 127, 255, cv2.THRESH_BINARY_INV)
    _, test_cu = cv2.threshold(img_test, 127, 255, cv2.THRESH_BINARY_INV)
    _, ref_bg = cv2.threshold(img_ref, 127, 255, cv2.THRESH_BINARY)
    _, test_bg = cv2.threshold(img_test, 127, 255, cv2.THRESH_BINARY)

    # 2. 提取基础实心差异
    raw_extra_cu = cv2.bitwise_and(test_cu, cv2.bitwise_not(ref_cu))
    raw_extra_bg = cv2.bitwise_and(test_bg, cv2.bitwise_not(ref_bg))

    # 设定形态学内核 (3x3对应半径1, 7x7对应半径3)
    kernel_3x3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    kernel_7x7 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))

    # 3. 提取 1 像素宽的内边缘轮廓 (模拟 HALCON 的 'margin')
    def get_inner_boundary(mask):
        eroded = cv2.erode(mask, kernel_3x3)
        return cv2.bitwise_and(mask, cv2.bitwise_not(eroded))

    ref_cu_boundary = get_inner_boundary(ref_cu)
    test_cu_boundary = get_inner_boundary(test_cu)
    ref_bg_boundary = get_inner_boundary(ref_bg)
    test_bg_boundary = get_inner_boundary(test_bg)

    # 构建参考图防线 (ToleranceRadius = 3)
    ref_cu_band = cv2.dilate(ref_cu_boundary, kernel_7x7)
    ref_bg_band = cv2.dilate(ref_bg_boundary, kernel_7x7)

    # 提取真缺陷种子并微量膨胀确保重叠
    seed_cu = cv2.bitwise_and(test_cu_boundary, cv2.bitwise_not(ref_cu_band))
    seed_cu_dilated = cv2.dilate(seed_cu, kernel_3x3)

    seed_bg = cv2.bitwise_and(test_bg_boundary, cv2.bitwise_not(ref_bg_band))
    seed_bg_dilated = cv2.dilate(seed_bg, kernel_3x3)

    # 4. 种子还原 (模拟 select_shape_proto 提取有效连通域)
    def seed_region_growing(raw_mask, seed_mask):
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(raw_mask, connectivity=8)
        valid_mask = np.zeros_like(raw_mask)
        
        for i in range(1, num_labels):
            # 过滤面积过小的基础噪点
            if stats[i, cv2.CC_STAT_AREA] >= 5:
                comp_mask = (labels == i).astype(np.uint8)
                # 若与种子有交集，则判定为真实缺陷实体
                if cv2.countNonZero(cv2.bitwise_and(comp_mask, seed_mask)) > 0:
                    valid_mask[labels == i] = 255
                    
        contours, _ = cv2.findContours(valid_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        return contours

    cu_candidates = seed_region_growing(raw_extra_cu, seed_cu_dilated)
    bg_candidates = seed_region_growing(raw_extra_bg, seed_bg_dilated)

    # 生成分类阶段所需的标签上下文
    _, ref_cu_labels = cv2.connectedComponents(ref_cu)
    _, ref_bg_labels = cv2.connectedComponents(ref_bg)
    _, test_cu_labels = cv2.connectedComponents(test_cu)
    _, test_bg_labels = cv2.connectedComponents(test_bg)

    context = {
        'shape': img_test.shape,
        'ref_cu_labels': ref_cu_labels,
        'ref_bg_labels': ref_bg_labels,
        'test_cu_labels': test_cu_labels,
        'test_bg_labels': test_bg_labels
    }

    return cu_candidates, bg_candidates, context

## 频域

In [ ]:
import cv2
import numpy as np

def extract_defect_candidates_1d(img_test, img_ref):
    """
    输入: 测试图, 模板图
    输出: 铜层异常轮廓, 背景层异常轮廓, 供分类使用的连通域上下文
    """
    
    # 参数定义区
    window_size = 21
    bump_threshold = 1.5
    min_area = 5
    closing_radius = 3

    # 1. 提取二值化区域
    _, test_cu = cv2.threshold(img_test, 127, 255, cv2.THRESH_BINARY_INV)
    _, ref_cu = cv2.threshold(img_ref, 127, 255, cv2.THRESH_BINARY_INV)
    
    _, test_bg = cv2.threshold(img_test, 127, 255, cv2.THRESH_BINARY)
    _, ref_bg = cv2.threshold(img_ref, 127, 255, cv2.THRESH_BINARY)

    # ==========================================
    # 1：提取绝对单像素宽度的参考图边缘
    # ==========================================
    ref_contours, _ = cv2.findContours(ref_cu, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_NONE)
    ref_edges = np.zeros_like(ref_cu)
    
    # 使用 thickness=1 描边，消除原有形态学操作导致的距离“死区”
    cv2.drawContours(ref_edges, ref_contours, -1, 255, 1)
    
    ref_edges_inv = cv2.bitwise_not(ref_edges)
    ref_dt = cv2.distanceTransform(ref_edges_inv, cv2.DIST_L2, 3)

    # 3. 提取测试图轮廓并准备画布
    test_contours, _ = cv2.findContours(test_cu, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_NONE)
    defect_edge_mask = np.zeros_like(img_test, dtype=np.uint8)
    kernel_1d = np.ones(window_size) / window_size

    # 4. 遍历提取一维距离信号
    for cnt in test_contours:
        cnt_points = cnt.squeeze(axis=1)
        if len(cnt_points.shape) == 1:
            cnt_points = cnt_points.reshape(1, 2)
            
        num_points = len(cnt_points)
        
        if num_points > window_size:
            xs = cnt_points[:, 0]
            ys = cnt_points[:, 1]
            
            # 精确查表测距
            distances = ref_dt[ys, xs]
            
            # 1D 滤波与高频提取
            pad_width = window_size // 2
            padded_dist = np.pad(distances, (pad_width, pad_width), mode='wrap')
            baseline_distances = np.convolve(padded_dist, kernel_1d, mode='valid')
            
            high_freq_diff = distances - baseline_distances
            
            # 锁定超过阈值的坐标
            defect_indices = np.where(high_freq_diff >= bump_threshold)[0]
            
            if len(defect_indices) > 0:
                defect_pts = cnt_points[defect_indices]
                defect_edge_mask[defect_pts[:, 1], defect_pts[:, 0]] = 255

    # 5. 形态学后处理与极性分离 
    # 废除 MORPH_CLOSE，改用强力膨胀 (Dilate)
    # 半径为 2 的膨胀核 (5x5)，确保 1 像素宽的边缘点能向外渗透进背景区，向内渗透进铜面区
    dilate_radius = 2
    kernel_dilate = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dilate_radius*2+1, dilate_radius*2+1))
    defect_edge_mask = cv2.dilate(defect_edge_mask, kernel_dilate)

    # 区别凸铜与凹铜（缺口）
    # 凸铜：边缘发生突变，且膨胀后的边缘带与“测试图多出的铜”重叠
    extra_cu_raw = cv2.bitwise_and(test_cu, cv2.bitwise_not(ref_cu))
    cu_defect_mask = cv2.bitwise_and(defect_edge_mask, extra_cu_raw)
    
    # 凹铜(缺口)：边缘发生突变，且膨胀后的边缘带与“测试图多出的背景”重叠
    extra_bg_raw = cv2.bitwise_and(test_bg, cv2.bitwise_not(ref_bg))
    bg_defect_mask = cv2.bitwise_and(defect_edge_mask, extra_bg_raw)

    # ==========================================
    # 替换过滤规则，规避格林公式面积漏洞
    # ==========================================
    def filter_contours(mask):
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        valid_cnts = []
        for c in contours:
            # 获取外接矩形的宽和高
            _, _, w, h = cv2.boundingRect(c)
            # 只要包围盒覆盖面积达标，或者绝对外围点数达标，即予以保留
            if (w * h >= min_area) or (len(c) >= min_area):
                valid_cnts.append(c)
        return valid_cnts

    cu_candidates = filter_contours(cu_defect_mask)
    bg_candidates = filter_contours(bg_defect_mask)

    # 7. 上下文信息构造
    _, ref_cu_labels = cv2.connectedComponents(ref_cu)
    _, ref_bg_labels = cv2.connectedComponents(ref_bg)
    _, test_cu_labels = cv2.connectedComponents(test_cu)
    _, test_bg_labels = cv2.connectedComponents(test_bg)

    context = {
        'shape': img_test.shape,
        'ref_cu_labels': ref_cu_labels,
        'ref_bg_labels': ref_bg_labels,
        'test_cu_labels': test_cu_labels,
        'test_bg_labels': test_bg_labels
    }

    return cu_candidates, bg_candidates, context

## 骨架加频域

In [ ]:
import cv2
import numpy as np

def extract_defect_candidates_merged(img_test, img_ref):
    """
    合并骨架检测与频域(1D)检测的结果，增加邻近聚合去重与图像物理边缘误检过滤。
    输入输出与原检测函数完全一致。
    """
    # 1. 分别调用两份原始检测代码
    cu_skel, bg_skel, context = extract_defect_candidates_skeleton(img_test, img_ref)
    cu_1d, bg_1d, _ = extract_defect_candidates_1d(img_test, img_ref)

    h, w = img_test.shape[:2]

    # 2. 定义边缘误检过滤函数：剔除紧贴图像最外圈物理边界的虚假候选
    def filter_out_border_contours(contours, height, width, margin=3):
        valid_contours = []
        for cnt in contours:
            # 计算轮廓的外接矩形
            x, y, bw, bh = cv2.boundingRect(cnt)
            # 如果外接矩形紧贴图像边界（超出margin范围），则认为是骨架边界截断导致的误检
            if x <= margin or y <= margin or (x + bw) >= (width - margin) or (y + bh) >= (height - margin):
                continue
            valid_contours.append(cnt)
        return valid_contours

    # 3. 定义高效率的“邻近聚合与去重”融合函数
    def merge_and_close_contours(contours_list_a, contours_list_b, canvas_shape):
        mask = np.zeros(canvas_shape, dtype=np.uint8)
        
        if contours_list_a:
            cv2.drawContours(mask, contours_list_a, -1, 255, thickness=cv2.FILLED)
        if contours_list_b:
            cv2.drawContours(mask, contours_list_b, -1, 255, thickness=cv2.FILLED)
            
        # 使用形态学闭运算或膨胀，将指向同一缺陷但微米级分离的碎片融合成一个整体
        # 半径为 2 的椭圆核可以把 3-5 像素以内的邻近残缺块安全桥接
        kernel_bridge = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        mask_closed = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_bridge)
        
        # 重新提取合并后的轮廓
        merged_contours, _ = cv2.findContours(mask_closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        return merged_contours

    # 4. 执行铜层与背景层的聚合与去重
    cu_raw_merged = merge_and_close_contours(cu_skel, cu_1d, (h, w))
    bg_raw_merged = merge_and_close_contours(bg_skel, bg_1d, (h, w))

    # 5. 执行边缘误检过滤
    cu_candidates = filter_out_border_contours(cu_raw_merged, h, w, margin=3)
    bg_candidates = filter_out_border_contours(bg_raw_merged, h, w, margin=3)

    return cu_candidates, bg_candidates, context

# 检测

In [ ]:

# ------------------------- 检测函数 -------------------------


def detect(test_img_path, temp_img_path, fc_name, dq=True):
    """
    对输入图片执行目标检测，返回检测结果列表
    返回格式： [{'bbox': [x1,y1,x2,y2], 'class': int}, ...]
    """
    img_ref = cv2.imread(temp_img_path, cv2.IMREAD_GRAYSCALE)
    img_test = cv2.imread(test_img_path, cv2.IMREAD_GRAYSCALE)

    if img_test is None or img_ref is None:
        raise ValueError("无法读取图像，请检查路径。")
    if dq:
        img_test = align_images_coarse(img_test, img_ref)

    if fc_name==r'extract_defect_candidates':
        cu_cands, bg_cands, context = extract_defect_candidates(img_test, img_ref)
    elif fc_name==r'extract_defect_candidates_skeleton':
        cu_cands, bg_cands, context = extract_defect_candidates_skeleton(img_test, img_ref)
    elif fc_name==r'extract_defect_candidates_drc':
        cu_cands, bg_cands, context = extract_defect_candidates_drc(img_test, img_ref)
    elif fc_name==r'extract_defect_candidates_seed':
        cu_cands, bg_cands, context = extract_defect_candidates_seed(img_test, img_ref)
    
    elif fc_name==r'extract_defect_candidates_skeleton1':
        cu_cands, bg_cands, context = extract_defect_candidates_skeleton1(img_test, img_ref)
    elif fc_name==r'extract_defect_candidates_1d':
        cu_cands, bg_cands, context = extract_defect_candidates_1d(img_test, img_ref)
        
    elif fc_name==r'extract_defect_candidates_merged':
        cu_cands, bg_cands, context = extract_defect_candidates_merged(img_test, img_ref)
        
    
    else:
        raise ValueError(r'检查你传入detect的函数名')

    # 第二步：分类
    final_results = classify_defects(cu_cands, bg_cands, context)
    return final_results


In [ ]:
import cv2
import os

# 颜色常量（BGR格式）
COLOR_RED    = (0, 0, 255)
COLOR_BLUE   = (255, 0, 0)
COLOR_GREEN  = (0, 255, 0)
COLOR_YELLOW = (0, 255, 255)

def draw_results(img_path, dets, gts, matches, unmatched_dets, unmatched_gts, output_path):
    """
    在图像上绘制检测与标注结果并保存
    - 漏检标注框（红色）
    - 标签匹配的检测框（蓝色）
    - 类型匹配（标签不同但类型一致）的检测框（绿色）
    - 误检检测框（黄色）
    同时在框旁标注类别数字
    """
    img = cv2.imread(img_path)
    if img is None:
        print(f"警告：无法读取图像 {img_path}，跳过绘制")
        return

    # 绘制所有漏检的标注框（红色）
    for idx in unmatched_gts:
        gt = gts[idx]
        x1, y1, x2, y2 = gt['bbox']
        cv2.rectangle(img, (x1, y1), (x2, y2), COLOR_RED, 2)
        cv2.putText(img, str(gt['class']), (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, COLOR_RED, 1)

    # 绘制正确检出的检测框，根据匹配类型区分颜色
    for det_idx, gt_idx in matches:
        det = dets[det_idx]
        gt = gts[gt_idx]
        x1, y1, x2, y2 = det['bbox']
        # 判断是否为标签匹配（class相同）
        if det['class'] == gt['class']:
            color = COLOR_BLUE   # 蓝色：标签匹配
        else:
            color = COLOR_GREEN  # 绿色：类型匹配（标签不同但类型一致）
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        cv2.putText(img, str(det['class']), (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

    # 绘制误检检测框（黄色）
    for idx in unmatched_dets:
        det = dets[idx]
        x1, y1, x2, y2 = det['bbox']
        cv2.rectangle(img, (x1, y1), (x2, y2), COLOR_YELLOW, 2)
        cv2.putText(img, str(det['class']), (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, COLOR_YELLOW, 1)

    # 保存结果图
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    cv2.imwrite(output_path, img)

# 主函数

In [ ]:
import time
import os
from collections import Counter
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    test_imgs = find_test_images(ROOT_DIR)
    print(f"共找到 {len(test_imgs)} 张 test 图片")

    # for fc_name in ['extract_defect_candidates', 'extract_defect_candidates_skeleton','extract_defect_candidates_1d','extract_defect_candidates_drc','extract_defect_candidates_seed']:
    for fc_name in ['extract_defect_candidates',
                    'extract_defect_candidates_1d','extract_defect_candidates_merged']:
        # 全局统计
        total_gt = 0
        total_tp = 0                # 总匹配数（标签匹配 + 类型匹配）
        total_label_match = 0       # 标签完全相同
        total_type_match = 0        # 类型相同但标签不同
        total_fp = 0
        total_fn = 0
        fn_class_counter = Counter()
        t1 = time.time()
        for test_path in test_imgs:
            temp_path = construct_temp_path(test_path)
            txt_path = construct_txt_path(test_path)

            dets = detect(test_path, temp_path, fc_name=fc_name, dq=False)
            # dets = detect(test_path, temp_path, fc_name=fc_name)

            gts = read_gt(txt_path)

            matches, unmatched_dets, unmatched_gts = match_detections(
                dets, gts,
                mode=MATCH_MODE,
                iou_thresh=IOU_THRESHOLD,
                center_dist_thresh=CENTER_DIST_THRESH,
                overlap_ratio_det=OVERLAP_RATIO_DET,
                overlap_ratio_gt=OVERLAP_RATIO_GT
            )

            total_gt += len(gts)
            total_tp += len(matches)
            total_fp += len(unmatched_dets)
            total_fn += len(unmatched_gts)

            # 分别统计标签匹配和类型匹配
            for i, j in matches:
                if dets[i]['class'] == gts[j]['class']:
                    total_label_match += 1
                else:
                    total_type_match += 1  # 类别不同但类型一致（因匹配条件保证）
            # 累加漏检类别分布
            for j in unmatched_gts:
                cls = gts[j]['class']
                fn_class_counter[cls] += 1
            # 绘制结果图（需修改 draw_results 以支持颜色区分）
            out_img_name = os.path.basename(test_path).replace('_test.jpg', '_result.jpg')
            out_img_path = os.path.join(OUTPUT_DIR, fc_name, out_img_name)
            draw_results(test_path, dets, gts, matches, unmatched_dets, unmatched_gts, out_img_path)

            print(f"处理: {test_path}")
            print(f"  标注数: {len(gts)}, 检测数: {len(dets)}, TP: {len(matches)}, FP: {len(unmatched_dets)}, FN: {len(unmatched_gts)}")
            print(f"  其中标签匹配: {sum(1 for i,j in matches if dets[i]['class'] == gts[j]['class'])}, "
                  f"类型匹配(标签不同): {sum(1 for i,j in matches if dets[i]['class'] != gts[j]['class'])}")

        t2 = time.time() - t1

        # 计算各种率
        recall = total_tp / total_gt if total_gt > 0 else 0.0
        if total_tp + total_fp > 0:
            false_positive_rate = total_fp / (total_tp + total_fp)
        else:
            false_positive_rate = 0.0

        label_recall = total_label_match / total_gt if total_gt > 0 else 0.0
        type_recall = total_type_match / total_gt if total_gt > 0 else 0.0

        # 输出全局统计
        print("\n" + "="*50)
        print("               全局统计结果")
        print("="*50)
        print(f"缺陷总数 (GT总数)        : {total_gt}")
        print(f"成功检出个数 (TP)        : {total_tp}")
        # print(f"  其中标签匹配 (class相同): {total_label_match}")
        # print(f"  其中类型匹配 (class不同但类型一致): {total_type_match}")
        print(f"误检个数 (FP)            : {total_fp}")
        print(f"漏检个数 (FN)            : {total_fn}")
        print(f"检出率 (Recall)          : {recall:.4f}")
        # print(f"  标签匹配召回率         : {label_recall:.4f}")
        # print(f"  类型匹配召回率         : {type_recall:.4f}")
        print(f"误检率 (FP/(TP+FP))      : {false_positive_rate:.4f}")
        print(f"用时: {t2:.2f} 秒")

        # 打印漏检类别分布
        print("\n漏检缺陷类别分布：")
        if fn_class_counter:
            for cls, count in sorted(fn_class_counter.items()):
                print(f"  类别 {cls}: {count} 次")
        else:
            print("  无漏检")

        print("="*50)

        # 保存统计报告（含漏检类别分布）
        report_path = os.path.join(OUTPUT_DIR, f"{fc_name}_statistics.txt")
        with open(report_path, 'w') as f:
            f.write("全局统计结果\n")
            f.write(f"缺陷总数: {total_gt}\n")
            f.write(f"成功检出个数: {total_tp}\n")
            # f.write(f"  其中标签匹配: {total_label_match}\n")
            # f.write(f"  其中类型匹配: {total_type_match}\n")
            f.write(f"误检个数: {total_fp}\n")
            f.write(f"漏检个数: {total_fn}\n")
            f.write(f"检出率: {recall:.4f}\n")
            # f.write(f"  标签匹配召回率: {label_recall:.4f}\n")
            # f.write(f"  类型匹配召回率: {type_recall:.4f}\n")
            f.write(f"误检率: {false_positive_rate:.4f}\n")
            f.write(f"用时: {t2:.2f} 秒\n")
            # 写入漏检类别分布
            f.write("\n漏检缺陷类别分布:\n")
            if fn_class_counter:
                for cls, count in sorted(fn_class_counter.items()):
                    f.write(f"  类别 {cls}: {count}\n")
            else:
                f.write("  无漏检\n")

        print(f"统计报告已保存至: {report_path}")

In [ ]:
main()

In [ ]:
import os
import sys
import pandas as pd
from pathlib import Path

# 文件名映射字典
FILENAME_MAP = {
    'extract_defect_candidates_1d_statistics.txt': '频域',
    'extract_defect_candidates_drc_statistics.txt': '形态学',
    'extract_defect_candidates_seed_statistics.txt': '轮廓',
    'extract_defect_candidates_skeleton_statistics.txt': '骨架',
    'extract_defect_candidates_statistics.txt': '区域',
}

# 类别映射字典
CATEGORY_MAP = {
    '1': '开路',
    '2': '短路',
    '3': '缺口',
    '4': '凸铜',
    '5': '残铜',
    '6': '针孔',
}

def parse_txt_stats(file_path: str) -> dict:
    """
    解析单个 txt 文件中的统计信息。
    期望每行格式为 "指标名: 数值"，冒号为英文半角。
    返回包含所有指标键值对的字典，数值自动转为 int 或 float。
    增加类别映射：将 "类别 数字" 转换为 "类别 中文名"。
    """
    stats = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or ':' not in line:
                continue
            key, value = line.split(':', 1)
            key = key.strip()
            value = value.strip()
            # 跳过值为空的行
            if not value:
                continue

            # 类别映射处理
            if key.startswith('类别') and ' ' in key:
                parts = key.split()
                if len(parts) == 2 and parts[1].isdigit():
                    num = parts[1]
                    if num in CATEGORY_MAP:
                        key = f'类别 {CATEGORY_MAP[num]}'

            # 尝试转换为数字，保留原字符串作为后备
            try:
                if '.' in value:
                    stats[key] = float(value)
                else:
                    stats[key] = int(value)
            except ValueError:
                stats[key] = value
    return stats

def collect_directory_stats(directory: str, output_excel: str = 'statistics.xlsx') -> None:
    """
    遍历目录下所有 .txt 文件，解析统计信息并保存至 Excel。
    文件名映射：将特定文件名替换为对应的中文算法名称。
    """
    all_rows = []
    txt_files = list(Path(directory).glob('*.txt'))
    if not txt_files:
        print(f"在目录 '{directory}' 中未找到任何 .txt 文件。")
        return

    for file_path in txt_files:
        try:
            stats = parse_txt_stats(file_path)
            if stats:  # 至少解析到一些内容
                # 文件名映射
                filename = file_path.name
                mapped_name = FILENAME_MAP.get(filename, filename)  # 无映射则保留原名
                stats['文件名'] = mapped_name
                all_rows.append(stats)
            else:
                print(f"警告: 文件 '{file_path}' 中未解析到任何有效指标行。")
        except Exception as e:
            print(f"错误: 处理文件 '{file_path}' 时发生异常: {e}")

    if not all_rows:
        print("没有成功解析任何文件的数据。")
        return

    # 构建 DataFrame，将 '文件名' 列置于最前
    df = pd.DataFrame(all_rows)
    cols = ['文件名'] + [c for c in df.columns if c != '文件名']
    df = df[cols]

    # 保存 Excel
    df.to_excel(output_excel, index=False, engine='openpyxl')
    print(f"成功汇总 {len(df)} 个文件的数据，已保存至 '{output_excel}'")

In [ ]:
target_dir = r'C:\Users\sangforsslvdi\Desktop\codetest\python\q1\output'
collect_directory_stats(target_dir)